# Acquisizione di tutte le membrane del campione (laser + oscilloscopio + motori PI)

Campione con **11 membrane lungo X e 9 lungo Y**, distanziate di **0.30 mm** in entrambe le direzioni. Dalla posizione della prima membrana (in alto a sinistra) il notebook le visita tutte **a meandro** e, su ognuna, fa **una scansione completa** del laser sincronizzata con lo scope (stessa sequenza di `acquisizione_matrice_xy.ipynb`), salvando un CSV per membrana.

- Motore **X** = `ASRL4` (COM4), motore **Y** = `ASRL5` (COM5). Limiti 0-150 mm.
- **Chiudi PI MicroMove** prima di partire: la porta seriale puo' essere usata da un solo programma.
- **Percorso:** si parte dalla prima membrana in alto a sinistra e si percorre la prima riga verso destra (lungo la riga inclinata); poi si scende di una membrana (Y decrescente) e si percorre la riga verso sinistra, poi si scende e si va di nuovo a destra, e cosi' via. Con 9 righe l'ultima va verso destra e finisce in basso a destra. Se "destra" o "giu'" sono invertiti sul tuo setup, cambia `DESTRA_E_X_CRESCENTE` e `GIU_E_Y_DECRESCENTE` in cella 1.
- **Scrivi in cella 1 la X della prima membrana** (`PRIMO_X_MM`): finche' e' `None` nessun motore si muove. `PRIMO_Y_MM` e `Y_FINE_PRIMA_RIGA_MM` sono la Y del centro della prima e dell'ultima (11a) membrana della prima riga.
- **Campione inclinato:** i centri delle membrane stanno su un reticolo ruotato. L'angolo si ricava dalla differenza di Y tra la prima e l'ultima membrana della prima riga (`sin(theta) = (Y_fine - Y_inizio) / (10 * passo)`) e si applica a tutte le righe. Con `COLONNE_RUOTATE_COME_LE_RIGHE = True` (rotazione rigida) anche scendendo di una riga la X si sposta (di `passo * sin(theta)`); con `False` le colonne restano verticali. **Controlla l'anteprima della cella 5, in particolare la posizione prevista delle membrane negli angoli in basso, e verifica a mano che sia giusta prima di dare il `si`.**
- Ogni membrana e' misurata in **un solo punto** (quello della sua posizione). Per misurare ogni membrana con una matrice di punti, vedi `acquisizione_matrice_xy.ipynb`.

Ordine: celle 1-4 una volta; la cella 5 mostra l'anteprima dei punti (**non muove niente**); la cella 6 li scansiona tutti e chiede conferma prima di muovere i motori.

Attenzione allo spazio su disco e al tempo: 99 acquisizioni da circa 240 MB sono circa 24 GB.


In [ ]:
# Cella 1 - configurazione e funzioni (scope, laser, griglia)
import os
import sys
import time
import datetime

import numpy as np
import pyvisa

OSC_VISA = "USB::0x0699::0x0408::C028813::INSTR"
OSC_TIMEOUT_MS = 30000

CHANNELS = ["CH1", "CH2"]

LASER_WAVELENGTH_START = 1520.0  # nm, come trigger_scan.m
LASER_WAVELENGTH_STOP = 1570.0   # nm, come trigger_scan.m

POLL_INTERVAL_S = 0.1
CHANNEL_READ_PAUSE_S = 0.1

# ---------------- membrane X-Y (in mm, limiti motori 0-150) ----------------
PRIMO_X_MM = None          # <-- SCRIVI QUI la X della prima membrana (in alto a sinistra)
PRIMO_Y_MM = 133.978       # Y del centro della prima membrana (in alto a sinistra)
Y_FINE_PRIMA_RIGA_MM = 134.078   # Y del centro dell'ultima (11a) membrana della prima riga: da qui si ricava l'inclinazione
COLONNE_RUOTATE_COME_LE_RIGHE = True   # True: rotazione rigida (scendendo di una riga anche X si sposta); False: colonne verticali
N_MEMBRANE_X = 11          # membrane lungo X
N_MEMBRANE_Y = 9           # membrane lungo Y
PASSO_MM = 0.30            # distanza tra membrane vicine, in X e in Y
DESTRA_E_X_CRESCENTE = True   # "destra" = X crescente
GIU_E_Y_DECRESCENTE = True    # "scendere" = Y decrescente; si parte dalla riga in alto

ASSESTAMENTO_S = 1.0           # pausa dopo il movimento, prima di acquisire
ATTESA_RITORNO_LASER_S = 5.0  # tra due scan, perché il laser torni a 1520 nm (CH1 basso)
RESETTA_CONTATORE_SCAN = True  # prima di ogni start: OUTP:SCAN:STOP + SOUR:WAVE:DESSCANS 1 (il laser potrebbe smettere di partire dopo N scan)
# --------------------------------------------------------------------------


def crea_cartella_run():
    oggi = datetime.date.today().strftime("%Y-%m-%d")
    base_folder = os.path.join(os.getcwd(), oggi)
    os.makedirs(base_folder, exist_ok=True)
    run_esistenti = [d for d in os.listdir(base_folder) if d.startswith("run_")]
    run_folder = os.path.join(base_folder, f"run_{len(run_esistenti) + 1:03d}")
    os.makedirs(run_folder)
    return run_folder


def configura_scope(osc):
    osc.write("HORizontal:MODE MANUAL")
    rec_len = int(osc.query("HORizontal:RECORDLENGTH?"))
    osc.write("DATA:START 1")
    osc.write(f"DATA:STOP {rec_len}")
    osc.write("DATA:WIDTH 1")
    osc.write("DATA:ENCdg ASCii")


def arm_acquisition(osc, timeout_s=120):
    osc.write("ACQ:STOPAFTER SEQUENCE")
    osc.write("ACQ:STATE RUN")
    # Lo scan parte solo quando lo scope e' READY (finito il pre-trigger).
    print("Attendo che lo scope finisca il pre-trigger (-> READY)...")
    t0 = time.time()
    ultimo = None
    while True:
        stato = osc.query("TRIGger:STATE?").strip().upper()
        if stato != ultimo:
            print(f"  stato trigger: {stato}")
            ultimo = stato
        if stato == "READY":
            break
        if time.time() - t0 > timeout_s:
            raise TimeoutError(
                f"Lo scope non e' arrivato a READY in {timeout_s} s "
                f"(stato trigger: {stato}, ACQ:STATE: {osc.query('ACQ:STATE?').strip()})"
            )
        time.sleep(POLL_INTERVAL_S)
    print("Scope pronto ad accettare il trigger.")


def wait_for_acquisition(osc):
    print("Attendo che l'oscilloscopio completi l'acquisizione triggerata...")
    while True:
        if int(osc.query("ACQ:STATE?")) == 0:
            print("Acquisizione completata!")
            break
        time.sleep(POLL_INTERVAL_S)


def acquire_channel(osc, source="CH1"):
    osc.write(f"DATA:SOURCE {source}")
    ymult = float(osc.query("WFMPRE:YMULT?"))
    yoff = float(osc.query("WFMPRE:YOFF?"))
    yzero = float(osc.query("WFMPRE:YZERO?"))
    print(f"{source}: YMULT={ymult}, YOFF={yoff}, YZERO={yzero}")
    raw = osc.query("CURVE?")
    ydata = np.array(raw.split(","), dtype=float)
    return (ydata - yoff) * ymult + yzero

In [ ]:
# Cella 2 - connessione all'oscilloscopio (una volta per sessione)
rm = pyvisa.ResourceManager()
osc = rm.open_resource(OSC_VISA)
osc.timeout = OSC_TIMEOUT_MS
configura_scope(osc)
print("Oscilloscopio connesso e configurato.")

In [ ]:
# Cella 3 - laser: carica la DLL e apre il dispositivo (una volta per sessione)
USBADDR = 1
DEVICE_ID = 0x100A  # hex2dec('100A')

dll_path = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "scripts", "laser", "UsbDllWrap.dll"))
if not os.path.isfile(dll_path):
    raise RuntimeError(f"UsbDllWrap.dll non trovata in {dll_path} (il notebook va aperto da Chiara_Bru/runs_oscilloscope)")
print(f"DLL trovata: {dll_path}")
sys.path.append(os.path.dirname(dll_path))

import clr
clr.AddReference("UsbDllWrap")
import Newport
from System.Text import StringBuilder

NP_USB = Newport.USBComm.USB()

esito = NP_USB.OpenDevices(DEVICE_ID)
print(f"OpenDevices({hex(DEVICE_ID)}) -> {esito}")
querydata = StringBuilder(64)


def laser_query(comando):
    esito = NP_USB.Query(USBADDR, comando, querydata)
    risposta = querydata.ToString()
    print(f"Query {comando!r} -> esito={esito}, risposta={risposta!r}")
    return risposta


LASER_COMANDI_CON_QUERY = True  # True: come i driver di riferimento (Query + lettura dell'OK)


def laser_write(comando):
    if LASER_COMANDI_CON_QUERY:
        # Con Write la risposta "OK" del laser resta non letta e si accumula,
        # sfasando le query successive e saturando la coda del laser.
        esito = NP_USB.Query(USBADDR, comando, querydata)
        print(f"Write {comando!r} -> esito={esito}, risposta={querydata.ToString()!r}")
    else:
        esito = NP_USB.Write(USBADDR, comando)
        print(f"Write {comando!r} -> esito={esito}")


laser_query("*IDN?")
laser_query("OUTP:STAT?")

In [ ]:
# Cella 4 - motori PI C-863: X = ASRL4, Y = ASRL5 (una volta per sessione)
MOTORE_X = "ASRL4::INSTR"
MOTORE_Y = "ASRL5::INSTR"
PI_BAUD = 19200
PI_AXIS = "1"
PI_TIMEOUT_MS = 2000
PI_TIMEOUT_MOVIMENTO_S = 60
PI_TOLLERANZA_MM = 0.01  # valore usato nella prova sul laboratorio (i motori arrivano entro circa 10 um)


def pi_apri(porta):
    inst = rm.open_resource(porta)
    inst.baud_rate = PI_BAUD
    inst.timeout = PI_TIMEOUT_MS
    inst.read_termination = "\n"
    inst.write_termination = "\n"
    return inst


def pi_query(inst, comando):
    return inst.query(comando).strip()


def pi_valore(inst, comando):
    """Estrae il numero da risposte tipo '1=+0012.3456'."""
    return float(pi_query(inst, comando).split("=")[-1])


def pi_limiti(inst):
    return pi_valore(inst, f"TMN? {PI_AXIS}"), pi_valore(inst, f"TMX? {PI_AXIS}")


def pi_muovi(inst, nome, target):
    """Movimento assoluto in mm, con controllo dei limiti prima di muovere."""
    tmin, tmax = pi_limiti(inst)
    if not (tmin <= target <= tmax):
        raise ValueError(f"Motore {nome}: target {target} mm fuori dai limiti {tmin}..{tmax} mm")
    inst.write(f"MOV {PI_AXIS} {target:.4f}")
    time.sleep(0.1)
    t0 = time.time()
    while True:
        in_target = pi_query(inst, f"ONT? {PI_AXIS}").endswith("1")
        pos = pi_valore(inst, f"POS? {PI_AXIS}")
        if in_target and abs(pos - target) <= PI_TOLLERANZA_MM:
            break
        if time.time() - t0 > PI_TIMEOUT_MOVIMENTO_S:
            raise TimeoutError(f"Motore {nome}: non arrivato a {target} mm (posizione {pos} mm)")
        time.sleep(0.1)
    errore = int(pi_query(inst, "ERR?"))
    if errore != 0:
        raise RuntimeError(f"Motore {nome}: ERR? = {errore} dopo il movimento a {target} mm")
    print(f"Motore {nome}: a {pos:.4f} mm")


motore_x = pi_apri(MOTORE_X)
motore_y = pi_apri(MOTORE_Y)

for nome, inst in (("X", motore_x), ("Y", motore_y)):
    errore_residuo = pi_query(inst, "ERR?")  # legge e azzera un eventuale errore vecchio
    tmin, tmax = pi_limiti(inst)
    print(f"Motore {nome}: {pi_query(inst, '*IDN?')}")
    print(f"   posizione {pi_valore(inst, f'POS? {PI_AXIS}')} mm | limiti {tmin}..{tmax} mm | "
          f"servo {pi_query(inst, f'SVO? {PI_AXIS}')} | referenziato {pi_query(inst, f'FRF? {PI_AXIS}')} | "
          f"errore residuo letto: {errore_residuo}")

In [ ]:
# Cella 5 - costruisce il percorso a meandro tenendo conto dell'inclinazione e mostra l'ANTEPRIMA (NON muove niente)
if PRIMO_X_MM is None or PRIMO_Y_MM is None or Y_FINE_PRIMA_RIGA_MM is None:
    raise ValueError("Scrivi in cella 1 PRIMO_X_MM (e controlla PRIMO_Y_MM e Y_FINE_PRIMA_RIGA_MM) e rilancia la cella 1")

n_x, n_y = int(N_MEMBRANE_X), int(N_MEMBRANE_Y)
sx = 1.0 if DESTRA_E_X_CRESCENTE else -1.0
sy = -1.0 if GIU_E_Y_DECRESCENTE else 1.0

# versore lungo la riga (dalla prima all'ultima membrana della prima riga) e versore verso la riga successiva
uy = (Y_FINE_PRIMA_RIGA_MM - PRIMO_Y_MM) / ((n_x - 1) * PASSO_MM)
if abs(uy) >= 1:
    raise ValueError("Y_FINE_PRIMA_RIGA_MM e' troppo lontana da PRIMO_Y_MM per il passo e il numero di membrane")
ux = sx * np.sqrt(1.0 - uy ** 2)
if COLONNE_RUOTATE_COME_LE_RIGHE:
    vx, vy = uy, -ux                 # perpendicolare alla riga
    if np.sign(vy) != sy:
        vx, vy = -uy, ux
else:
    vx, vy = 0.0, sy                 # colonne verticali

PUNTI = []
for j in range(n_y):
    colonne = range(n_x) if j % 2 == 0 else range(n_x - 1, -1, -1)       # meandro
    for i in colonne:
        PUNTI.append((float(PRIMO_X_MM + PASSO_MM * (i * ux + j * vx)),
                      float(PRIMO_Y_MM + PASSO_MM * (i * uy + j * vy))))

print(f"{n_x} x {n_y} membrane, passo {PASSO_MM} mm, percorso a meandro")
print(f"Inclinazione del campione: {np.degrees(np.arcsin(uy)):.3f} gradi")
print(f"  a ogni membrana verso destra: dX = {PASSO_MM * ux:+.5f} mm, dY = {PASSO_MM * uy:+.5f} mm")
print(f"  a ogni riga verso il basso:   dX = {PASSO_MM * vx:+.5f} mm, dY = {PASSO_MM * vy:+.5f} mm")
print("Angoli previsti (verifica almeno quelli in basso a mano):")
for nome, (i, j) in (("alto-sinistra", (0, 0)), ("alto-destra", (n_x - 1, 0)),
                     ("basso-sinistra", (0, n_y - 1)), ("basso-destra", (n_x - 1, n_y - 1))):
    print(f"  {nome:15s}: X={PRIMO_X_MM + PASSO_MM * (i * ux + j * vx):.4f}  Y={PRIMO_Y_MM + PASSO_MM * (i * uy + j * vy):.4f}")

tutti_x = np.array([p[0] for p in PUNTI])
tutti_y = np.array([p[1] for p in PUNTI])
for nome, inst, valori in (("X", motore_x, tutti_x), ("Y", motore_y, tutti_y)):
    tmin, tmax = pi_limiti(inst)
    if valori.min() < tmin or valori.max() > tmax:
        raise ValueError(f"Punti {nome} fuori dai limiti {tmin}..{tmax} mm (da {valori.min():.3f} a {valori.max():.3f})")

print(f"{len(PUNTI)} punti, limiti rispettati. Si parte da X={PUNTI[0][0]:.3f}, Y={PUNTI[0][1]:.3f} e si finisce a X={PUNTI[-1][0]:.3f}, Y={PUNTI[-1][1]:.3f}")
for k, (x, y) in enumerate(PUNTI):
    print(f"  {k + 1:3d}: X={x:.4f}  Y={y:.4f}")
print(f"Spazio su disco stimato: circa {len(PUNTI) * 0.24:.1f} GB")


In [ ]:
# Cella 6 - ESEGUE la griglia: per ogni punto muove X e Y, poi una scansione completa
if input(f"Muovere i motori ed eseguire {len(PUNTI)} scansioni? scrivi 'si' per confermare: ").strip().lower() != "si":
    raise RuntimeError("Annullato: nessun movimento eseguito.")

run_folder = crea_cartella_run()
print(f"I dati saranno salvati in: {run_folder}")
registro_path = os.path.join(run_folder, "griglia.csv")
with open(registro_path, "w") as f:
    f.write("indice,x_mm,y_mm,file\n")

t0p = time.time()


def tp(msg):
    print(f"   [{time.time() - t0p:6.1f} s] {msg}")


try:
    for idx, (x, y) in enumerate(PUNTI):
        print(f"\n=== Punto {idx + 1}/{len(PUNTI)}: X={x:.3f} mm, Y={y:.3f} mm ===")
        t0p = time.time()
        pi_muovi(motore_x, "X", x)
        pi_muovi(motore_y, "Y", y)
        tp("motori in posizione")
        time.sleep(ASSESTAMENTO_S)
        tp("assestamento finito")

        # stessa sequenza dell'altro notebook (acquisizione_laser_scope.ipynb)
        laser_write(f"SOUR:WAVE:START {LASER_WAVELENGTH_START}")
        laser_write(f"SOUR:WAVE:STOP {LASER_WAVELENGTH_STOP}")
        if RESETTA_CONTATORE_SCAN:
            laser_write("OUTP:SCAN:STOP")
            laser_write("SOUR:WAVE:DESSCANS 1")

        arm_acquisition(osc)
        tp("scope READY")
        laser_write("OUTP:SCAN:START")
        tp("start inviato al laser")

        wait_for_acquisition(osc)
        tp("acquisizione completata")
        time.sleep(1.0)

        data = {}
        for ch in CHANNELS:
            print(f"Leggo {ch}...")
            data[ch] = acquire_channel(osc, ch)
            time.sleep(CHANNEL_READ_PAUSE_S)
        tp("canali letti")

        nome_file = f"punto_{idx:03d}_x{x:.3f}_y{y:.3f}.csv"
        min_len = min(len(v) for v in data.values())
        all_cols = np.column_stack([data[ch][:min_len] for ch in CHANNELS])
        np.savetxt(os.path.join(run_folder, nome_file), all_cols, delimiter=",", header=",".join(CHANNELS), comments="")
        with open(registro_path, "a") as f:
            f.write(f"{idx},{x:.4f},{y:.4f},{nome_file}\n")
        print(f"Salvato: {nome_file}")
        tp("file salvato")

        if idx < len(PUNTI) - 1:
            print(f"Attendo {ATTESA_RITORNO_LASER_S:.0f} s che il laser torni a {LASER_WAVELENGTH_START} nm...")
            time.sleep(ATTESA_RITORNO_LASER_S)
            tp("fine attesa laser")
except BaseException:
    print("\nInterrotto: provo a fermare lo scan del laser.")
    try:
        laser_write("OUTP:SCAN:STOP")
    except Exception as e:
        print(f"Impossibile fermare lo scan: {e}")
    raise

print(f"\nGriglia completata: {len(PUNTI)} punti in {run_folder}")